In [ ]:



# import gtfs_kit as gk
# from shapely import wkt
# import csv
# import zipfile
# from datetime import date
# from datetime import datetime
# import shutil
# import numpy as np
# import matplotlib.pyplot as plt
# import plotly
# import plotly.graph_objects as go
# import plotly.express as px
# import plotly.io as pio
# from plotly.offline import plot
# import plotly.subplots as sp
# from plotly.subplots import make_subplots


## Packages ---

from pathlib import Path
import pandas as pd
import geopandas as gpd
import os
import re
from arcgis.features import GeoAccessor, GeoSeriesAccessor
from IPython.display import display
# import pdb; pdb.set_trace()




In [ ]:


# Read in population table
# Reshape table from long to wide so that there is only one row per block group and one column for each population by race/eth for the latest year
# Read in block group shapefile
# Merge on block group GEOID, the population table onto the block group shapefile
# Export as shapefile
# Export as filegdb


import warnings
warnings.filterwarnings('ignore')

def re_remove_post(x, exp = '.'):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]



print('Importing/processing excel or csv file to merge onto the geospatial layer...')
path_in = Path(r'C:\Users\jfontes\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Products\CERF\We Prosper Together\Population')
wkbook = 'Pop_3 Block Groups ACS5_ValleyVision.xlsx'
sheet_name = 'Block Groups'
file_in = path_in / wkbook
df = pd.read_excel(file_in, sheet_name=sheet_name)


df['Block Group ID'] = df['Block Group ID'].fillna(0)
df['Block Group ID'] = df['Block Group ID'].astype(str).apply(re_remove_post)


df['State FIPS'    ] = df['State FIPS'    ].astype(str).apply('{:0>2}'.format)
df['County FIPS'   ] = df['County FIPS'   ].astype(str).apply('{:0>3}'.format)
df['Tract ID'      ] = df['Tract ID'      ].astype(str).apply('{:0>6}'.format)
df['Block Group ID'] = df['Block Group ID'].astype(str)

df['Census Tract'] = df['State FIPS'] + df['County FIPS'] + df['Tract ID']
df['GEOID'       ] = df['State FIPS'] + df['County FIPS'] + df['Tract ID'] + df['Block Group ID']
df['GEOID'] = df['GEOID'].astype('int64')




df = df.sort_values(['GEOID', 'Year'], ascending=[True,False])
df = df.drop_duplicates(['GEOID', 'Race_Ethnicity'])
df = df[['GEOID', 'Race_Ethnicity', 'Population']]

df = df.pivot_table(index='GEOID', columns='Race_Ethnicity', values='Population').reset_index()



print('Import/processing geospatial layer...')
path_shp = Path(r'I:\Projects\Josh\Geospatial Data\TIGER\geojson')
shpname = 'tl_2020_sacog_bg.geojson'
file_shp = path_shp / shpname
gdf_bg = gpd.read_file(file_shp)


gdf_bg = gdf_bg[['GEOID', 'geometry']]

gdf_bg = gdf_bg.merge(df, on='GEOID', how='left')

gdf_bg.columns = [x.lower() for x in gdf_bg.columns]
gdf_bg.columns = [re.sub('[^\\w\\s]', '_', col.strip()) for col in gdf_bg.columns]
gdf_bg.columns = [re.sub('[\s+]'    , '_', col.strip()) for col in gdf_bg.columns]
gdf_bg.columns = [re.sub('\\?'      , '' , col.strip()) for col in gdf_bg.columns]

gdf_bg = gdf_bg[['geoid', 'all', 'asian__nh_', 'black_or_african_american__nh_', 'hispanic_or_latino', 'white__nh_', 'geometry']]
gdf_bg.columns = ['geoid', 'all', 'asian_nh', 'black_nh', 'hispanic_nh', 'white_nh', 'geometry']
gdf_bg = gdf_bg.fillna(0)




print('Exporting to shp...')
path_out = Path(r'I:\Projects\Josh\Regional Monitoring\Accessibility\shp\ValleyVision')
shp_out = "ValleyVision_pop3"
file_shp = path_out / shp_out
os.makedirs(file_shp, exist_ok=True)
file_shp_out = file_shp / f'{shp_out}.shp'
gdf_bg.to_file(file_shp)




print('Exporting to file geodatabase...')

gdf_data = gdf_data.to_crs(crs)
sdf_data = GeoAccessor.from_geodataframe(gdf_data, column_name='geometry')

print(f'Exporting feature class {file_to_convert} to the file geodatabase {file_gdb}...'); print(); print()
sdf_data.spatial.to_featureclass(location=file_fc)
print(f'Successfully exported to the following location: {file_gdb}'); print(); print()




In [ ]:


# # updates by josh
# def amtrak(op_dir):

#     gtfs = gk.read_feed(op_dir, dist_units='mi')

#     bbox = { # ValleyVision bounding box
#         'point': ['northwest', 'southwest', 'northeast', 'southeast'],
#         'coordinates': ['POINT(-122.9 39.7)', 'POINT(-122.9 37.9)', 'POINT(-119.7 39.7)', 'POINT(-119.7 37.9)']
#     }

#     df_bbox = pd.DataFrame(bbox)

#     df_bbox["coordinates"] = gpd.GeoSeries.from_wkt(df_bbox["coordinates"])
#     gdf_area = gpd.GeoDataFrame(df_bbox, geometry="coordinates", crs='EPSG:4326')

#     gtfs = gtfs.restrict_to_area(gdf_area)

#     gtfs_tables = ['agency', 'calendar', 'calendar_dates', 'feed_info', 'routes', 'stop_times', 'stops', 'trips'] # Add other required and optional tables

#     with zipfile.ZipFile(op_dir, 'w', zipfile.ZIP_DEFLATED) as zf:
#         for table_name in gtfs_tables:
#             df = getattr(gtfs, table_name) # Access the DataFrame dynamically
#             if df is not None:
#                 file_name = f"{table_name}.txt"
#                 df.to_csv(file_name, index=False) # Save as .txt
#                 zf.write(file_name)  # Add to the zip file


In [ ]:


path_amtrak = Path(r'I:\Projects\Josh\Geospatial Data\GTFS\datemod_versions\improved-gtfs-amtrak')

file_stops = path_amtrak / 'stops.txt'
df_stops = pd.read_csv(file_stops, sep=',')

df_stops



In [ ]:
df_stops[df_stops['stop_id'] == '597-RUG']

In [ ]:


list_folders = [folder for folder in path_gtfs.iterdir() if folder.is_dir()]
list_folders = [folder for folder in list_folders if '.zip' not in str(folder)]
list_folders

list_gdf = []

for folder in list_folders:

    txt_stops = [txt_stops for txt_stops in folder.iterdir() if 'stops' in str(txt_stops)][0]
    df_stops = pd.read_csv(txt_stops, sep=',')

    print(folder.stem)
    print(df_stops['stop_url'].unique())
    display(df_stops.head())

